# Histogram delta compression — wire bytes and throughput

Reads the output of `run_histogram_delta_compression.sh` and draws the two figures:

1. **Bytes on the wire** per payload representation (rate-limited run — every variant ingests the same
   tuples, so a difference in bytes is a difference in representation).
2. **Ingest throughput** per representation (unthrottled Memory-source run, median of repetitions).

The two come from separate runs by necessity: equalising ingest is what makes the byte counts
comparable, and *not* equalising it is what makes the throughput meaningful.

Figures are drawn without a title or subtitle — they sit under a paper caption. What stays on the
canvas is what a caption cannot replace: axis labels, category labels and per-bar values.

In [ ]:
import os
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

# Point this at the run you want to plot (the OUT_ROOT that run_histogram_delta_compression.sh printed).
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", "../plots/histogram_delta_run"))
OUT_DIR = Path(os.environ.get("FIGURE_DIR", RESULTS_DIR / "figures"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS_DIR.resolve())
print("figures:", OUT_DIR.resolve())

## Styling

Okabe-Ito colourblind-safe palette in a fixed order, matching the other benchmark suites. The
assignment is deliberate: hue family = plan shape (blue = full synopsis, orange = delta), lighter or
darker within the family = whether the blob is zstd-compressed.

Palette check on the adjacent-pair list (the one bars use), computed rather than eyeballed: lightness
band PASS, chroma floor PASS, CVD separation PASS (worst 13.1 ΔE deutan against a target of 8.0),
normal-vision separation PASS (worst 15.6 ΔE against a floor of 15.0). Contrast against the surface
WARNs for the two lightest bars, which is discharged by never encoding identity in colour alone —
every bar carries its category as a tick label and its value as a direct label.

In [ ]:
BLACK, ORANGE, SKY, GREEN, YELLOW, BLUE, VERMILLION, PURPLE = (
    "#000000", "#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7")
GRID, INK, MUTED = "#dddddd", "#222222", "#666666"

VARIANT_ORDER = ["split", "split_zstd", "delta", "delta_zstd"]
VARIANT_COLOR = {"split": BLUE, "split_zstd": SKY, "delta": VERMILLION, "delta_zstd": ORANGE}
VARIANT_LABEL = {
    "split": "split\nfull synopsis",
    "split_zstd": "split + zstd\ncompressed synopsis",
    "delta": "delta\nsparse delta",
    "delta_zstd": "delta + zstd\ncompressed delta",
}
BASELINE = "split"  # every comparison is against the uncompressed synopsis


def style(ax):
    """Recessive chrome: hairline horizontal grid, no top/right spines, muted ticks."""
    ax.grid(True, axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.grid(False, axis="x")  # vertical gridlines say nothing about nominal categories
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(MUTED)
    ax.tick_params(colors=MUTED)


def ordered(variants):
    present = set(variants)
    return [v for v in VARIANT_ORDER if v in present]


def bars(ax, variants, values, notes):
    """Vertical bars, linear axis, one direct label per bar.

    Linear on purpose: bar height must stay proportional to the value it encodes, so a log axis is not
    an option even though the values span a wide range.
    """
    xpos = range(len(variants))
    ax.bar(xpos, values, color=[VARIANT_COLOR[v] for v in variants], width=0.62, zorder=3)
    for x, v, note in zip(xpos, values, notes):
        ax.annotate(note, (x, v), color=INK, fontsize=9, ha="center", va="bottom",
                    xytext=(0, 5), textcoords="offset points", linespacing=1.35)
    ax.set_xticks(list(xpos))
    ax.set_xticklabels([VARIANT_LABEL[v] for v in variants], fontsize=9, color=INK)
    ax.set_ylim(0, max(values) * 1.22)  # headroom for the two-line labels

## Figure 1 — bytes on the wire

`total_bytes` is the root container's eth0 RX over the run, i.e. everything that actually crossed the
network to reach the store owner.

In [ ]:
wire = pd.read_csv(RESULTS_DIR / "bytes" / "summary.csv")
wire = wire[wire["variant"].isin(VARIANT_ORDER)]
variants = ordered(wire["variant"])
mb = {v: wire.loc[wire["variant"] == v, "total_bytes"].iloc[0] / 1e6 for v in variants}
base = mb[BASELINE]

notes = []
for v in variants:
    note = f"{mb[v]:,.1f} MB"
    if v != BASELINE:
        # Say which direction, so a variant above the baseline never reads as a saving.
        note += f"\n{base / mb[v]:.1f}x less" if mb[v] < base else f"\n{mb[v] / base:.1f}x more"
    notes.append(note)

fig, ax = plt.subplots(figsize=(1.55 * len(variants) + 1.6, 5.2))
style(ax)
bars(ax, variants, [mb[v] for v in variants], notes)
ax.set_ylabel("bytes received (MB)", color=INK)
fig.tight_layout()
fig.savefig(OUT_DIR / "wire_bytes.png", dpi=150)
print(pd.DataFrame({"variant": variants, "MB": [round(mb[v], 2) for v in variants],
                    "vs split": [round(base / mb[v], 2) for v in variants]}).to_string(index=False))

## Figure 2 — ingest throughput

`mean_tps` is the engine's own throughput listener, averaged over the active span (first to last
non-zero sample, interior zeros kept so stalls count). Grouped by local query id and taken over the
busiest stream, so a delta run's GEN ingest rate is not averaged with its RESOLVER's
one-blob-per-window rate.

Median across repetitions rather than mean: one run that hit a slow compile is an outlier and should
not drag the reported figure. The observed spread is printed alongside.

In [ ]:
runs = sorted((RESULTS_DIR / "throughput").glob("run*/summary.csv"))
print(f"{len(runs)} repetition(s):", [p.parent.name for p in runs])

tp = pd.concat([pd.read_csv(p).assign(run=p.parent.name) for p in runs], ignore_index=True)
tp = tp[tp["variant"].isin(VARIANT_ORDER)]

agg = tp.groupby("variant")["mean_tps"].agg(["median", "min", "max"]) / 1e6
variants = ordered(agg.index)
mtps = {v: agg.loc[v, "median"] for v in variants}
base_tps = mtps[BASELINE]

notes = []
for v in variants:
    note = f"{mtps[v]:.2f} M/s"
    if v != BASELINE:
        pct = (mtps[v] / base_tps - 1) * 100
        note += f"\n{abs(pct):.0f}% {'faster' if pct >= 0 else 'slower'}"
    notes.append(note)

fig, ax = plt.subplots(figsize=(1.55 * len(variants) + 1.6, 5.2))
style(ax)
bars(ax, variants, [mtps[v] for v in variants], notes)
ax.set_ylabel("ingest throughput (M tuples/s)", color=INK)
fig.tight_layout()
fig.savefig(OUT_DIR / "throughput.png", dpi=150)

report = agg.loc[variants].copy()
report["spread %"] = ((report["max"] - report["min"]) / report["max"] * 100).round(1)
report["vs split %"] = ((report["median"] / base_tps - 1) * 100).round(0)
print(report.round(3).to_string())

## Table view

Every number behind both figures, so nothing is reachable only by reading a bar height.

In [ ]:
table = (wire[["variant", "total_bytes"]]
         .assign(MB=lambda d: (d["total_bytes"] / 1e6).round(2))
         .drop(columns="total_bytes")
         .merge(agg["median"].rename("MTup_s").round(2), on="variant"))
table["bytes_vs_split"] = (base / table["variant"].map(mb)).round(2)
table["tps_vs_split_%"] = ((table["variant"].map(mtps) / base_tps - 1) * 100).round(0)
table = table.set_index("variant").loc[variants]
display(table)
table.to_csv(OUT_DIR / "figures_table.csv")
print("wrote", OUT_DIR / "figures_table.csv")